# SGD, Adam, AdamW : pourquoi le même gradient ne suffit pas

Comparer les règles de mise à jour avant de changer l’optimiseur d’une expérience.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#optimiseurs-regularisation-pratique).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Votre boucle apprend déjà avec SGD. Avant de passer à Adam par habitude, isolons ce qui changerait : le gradient peut rester identique, mais son utilisation diffère.

Nous repartons d’un poids égal à 1, avec L=(poids−2)². Sa dérivée vaut −2. Le taux lr=0,1 et la décroissance weight_decay=0,1 sont imposés pour comparer un seul pas.

Pour SGD sans momentum, la décroissance ajoute 0,1×poids au gradient. La direction vaut donc −1,9 et le poids devient 1−0,1×(−1,9)=1,19. poids.grad reste pourtant égal à −2 : c’est la règle de mise à jour qui ajoute ce terme.

| Optimiseur | Gradient de la perte | Poids obtenu |
| --- | --- | --- |
| SGD | −2 | 1,19 |
| Adam | −2 | 1,10 |
| AdamW | −2 | 1,09 |

Ces sorties utilisent les paramètres par défaut de PyTorch pour chaque famille, avec le taux et la décroissance indiqués.

Adam mémorise des moyennes du gradient et de son carré, puis corrige leur initialisation pour adapter le pas. Dans sa formulation par défaut ici, la décroissance entre dans le gradient utilisé par ces moyennes.

AdamW sépare cette décroissance : il contracte d’abord le poids de 0,01 dans ce cas, puis ajoute environ 0,1 par l’adaptation Adam. On arrive à 1,09. Un même weight_decay ne désigne donc pas le même calcul.

Ce seul pas ne permet pas de choisir un gagnant. Pour le classifieur, comparez la validation et les taux adaptés à chaque règle. Pour continuer Adam demain, il faudra aussi conserver son historique.

## Démonstration 1 · Recommencer chaque essai au même point

Comparer une mise à jour, pas trois entraînements successifs.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch


### Calculer l’état suivant

La boucle parcourt des constructeurs ; chaque passage crée un poids neuf.

In [4]:
resultats = {}


In [5]:
observer('resultats', resultats)

resultats : clés []


### Répéter la même étape sur les données

La liste contient le seul paramètre ajustable. Les deux valeurs de réglage sont identiques entre essais.

backward calcule le gradient de la perte ; item ne fait qu’en lire la valeur.

step combine la pente et l’état de la règle choisie.

In [6]:
for famille in [torch.optim.SGD, torch.optim.Adam, torch.optim.AdamW]:
    poids = torch.tensor(1., requires_grad=True)
    optimiseur = famille([poids], lr=0.1, weight_decay=0.1)
    perte = (poids - 2.) ** 2
    perte.backward()
    gradient = poids.grad.item()
    optimiseur.step()
    resultats[famille.__name__] = (gradient, poids.item())
    print(famille.__name__, 'gradient :', gradient, 'poids :', round(poids.item(), 2))

SGD gradient : -2.0 poids : 1.19
Adam gradient : -2.0 poids : 1.1
AdamW gradient : -2.0 poids : 1.09


In [7]:
assert bool(all(g == -2 for g, w in resultats.values())), 'optimiseurs-regularisation-pratique/main: all(g == -2 for g, w in resultats.values())'
assert bool(abs(resultats['SGD'][1] - 1.19) < 1e-6), "optimiseurs-regularisation-pratique/main: abs(resultats['SGD'][1] - 1.19) < 1e-6"
assert bool(abs(resultats['Adam'][1] - 1.1) < 1e-6 and abs(resultats['AdamW'][1] - 1.09) < 1e-6), "optimiseurs-regularisation-pratique/main: abs(resultats['Adam'][1] - 1.1) < 1e-6 and abs(resultats['AdamW'][1] - 1.09) < 1e-6"
print("Contrôles numériques de la démonstration réussis.")

Contrôles numériques de la démonstration réussis.


La dérivée ne définit pas à elle seule la trajectoire d’apprentissage.

## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

AdamW reçoit un poids 1, un gradient nul, lr=0,1 et weight_decay=0,1. Quel poids suit le pas ?

In [8]:
exo_p = torch.nn.Parameter(torch.tensor(1.))
exo_opt = torch.optim.AdamW([exo_p],lr=.1,weight_decay=.1)
exo_p.grad = torch.zeros_like(exo_p)
exo_opt.step()
exo_poids = ...  # Remplacer les points par votre réponse.
if exo_poids is Ellipsis or exo_poids is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_poids)-exo_p.item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

Avec un gradient nul explicite, la décroissance découplée applique ici le facteur 1−lr×weight_decay.

</details>

### Exercice 2

SGD reçoit un poids 1, un gradient 2, lr=0,1 et weight_decay=0,1. Quel poids suit le pas ?

In [9]:
exo_p = torch.nn.Parameter(torch.tensor(1.))
exo_opt = torch.optim.SGD([exo_p],lr=.1,weight_decay=.1)
exo_p.grad = torch.tensor(2.)
exo_opt.step()
exo_poids = ...  # Remplacer les points par votre réponse.
if exo_poids is Ellipsis or exo_poids is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_poids)-exo_p.item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

La contribution de décroissance vaut 0,1×1. Soustrayez 0,1×(2+0,1).

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_p = torch.nn.Parameter(torch.tensor(1.))
exo_opt = torch.optim.AdamW([exo_p],lr=.1,weight_decay=.1)
exo_p.grad = torch.zeros_like(exo_p)
exo_opt.step()
exo_poids = 0.99
if exo_poids is Ellipsis or exo_poids is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_poids)-exo_p.item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_p = torch.nn.Parameter(torch.tensor(1.))
exo_opt = torch.optim.SGD([exo_p],lr=.1,weight_decay=.1)
exo_p.grad = torch.tensor(2.)
exo_opt.step()
exo_poids = 0.79
if exo_poids is Ellipsis or exo_poids is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_poids)-exo_p.item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [12]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()